# CpG Island Detection

Scans the upstream sequences produced by `downloader.ipynb` using a sliding
window to identify CpG islands (CGIs), based on the classic Gardiner-Garden
& Frommer (1987) criteria.

**Input**: `dataset/processed/{species}_mrna.csv`

**Output**: `dataset/processed/{species}_islands.csv`


### CpG rules
Since there are some conditions to be a CpG island, check if given sequence fulfills those.

1. length > 200bp
2. GC content > 50%
3. observed / expected == (# of CpG * length) / (# or C * # of G) > 0.6

If there are more than 100bp of sequence that does not fultill these conditions, former and later part of that sequence will be another island.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

In [ ]:
# CpG island criteria (Gardiner-Garden & Frommer, 1987)
WINDOW = 200        # sliding window size (bp) — corresponds to "length > 200bp"
THRESHOLD = 100      # max consecutive bp allowed to fail criteria before splitting into a new island
MIN_LEN = 200        # minimum final island length (bp)
MIN_CONTENT = 0.5    # minimum GC content (fraction)
MIN_RATIO = 0.6      # minimum observed/expected CpG ratio

In [ ]:
def isCpG(content):
  # Check whether a window satisfies all three CGI criteria.
    if ((content['c'] + content['g']) / content['len'] >= MIN_CONTENT and
        content['c'] != 0 and content['g'] != 0 and
        (content['cpg'] * content['len']) / (content['c'] * content['g']) >= MIN_RATIO):
        return True
    return False

In [ ]:
def island_finder(fulfill):
    """
    Merge a boolean list (per-window CGI pass/fail) into island coordinates.

    Consecutive windows are grouped into the same island as long as no more
    than THRESHOLD consecutive bp fail the CGI criteria (per Gardiner-Garden
    & Frommer, gaps up to 100bp are tolerated within a single island).
    Islands shorter than MIN_LEN after merging are discarded.

    Args:
        fulfill (list[bool]): per-window CGI pass/fail flags along the sequence

    Returns:
        list[list[int, int]] | None: [start, end] window-index pairs for each
            island found, or None if no window passed the criteria
    """

    result = [] # start, end index of CpG island
    true_len = 0
    false_len = 0

    if True not in fulfill:
        return None

    current_start = fulfill.index(True)
    i = current_start
    while i < len(fulfill):
        if fulfill[i]:
            true_len += 1
            false_len = 0
        else:
            false_len += 1

        #if there are more than 100bp(threshold) of sequence that does not fultill these conditions
        if false_len >= THRESHOLD:
            #Save index of current CpG island
            if i-THRESHOLD - current_start + WINDOW >= MIN_LEN:
                result.append([current_start, i-THRESHOLD])
            if True in fulfill[i:]:
                i += fulfill[i:].index(True)
                current_start = i
                true_len = 0
                false_len = 0
                continue
            else: #End if there are no true sequence left
                return result

        i+=1

    last_index = len(fulfill) - false_len - 1
    if last_index - current_start + WINDOW >= MIN_LEN:
      result.append([current_start, last_index])
    return result

In [ ]:
def content_alter(outgoing, incoming):
    """
    Compute the incremental change in C/G/CpG counts when the sliding
    window shifts by one base, without recounting the entire window
    (O(1) update instead of O(WINDOW) per step).

    Args:
        outgoing (str): 2-character string, the base leaving the window
                        plus its former neighbor (used to detect a CpG
                        pair being lost)
        incoming (str): 2-character string, the new base entering the
                        window plus its new neighbor (used to detect a
                        new CpG pair being gained)

    Returns:
        dict: {'c': delta C count, 'g': delta G count, 'cpg': delta CpG count}
    """
    alter = dict()

    alter['c'] = incoming[1:].count('C') - outgoing[:-1].count('C')
    alter['g'] = incoming[1:].count('G') - outgoing[:-1].count('G')

    alter['cpg'] = 0
    if 'CG' in incoming:
        alter['cpg'] += incoming.count('CG')
    if 'CG' in outgoing:
        alter['cpg'] -= outgoing.count('CG')

    return alter

In [ ]:
def findCpG(data):
    """
    Scan each sequence in `data` with a sliding window and extract all
    regions meeting the CpG island criteria.

    Args:
        data (pd.DataFrame): must contain 'id', 'chrom', 'seq' columns
                              (output of downloader.ipynb)

    Returns:
        list[list]: rows of [id, chrom, start, end, seq, gc-content,
                    length, obs/exp] for each detected CpG island
    """
    temp_islands = []

    for entry in tqdm(data.itertuples()):
        seq = entry.seq
        fulfill = []
        content = {
            'c': seq[:WINDOW].count('C'),
            'g': seq[:WINDOW].count('G'),
            'cpg': seq[:WINDOW].count('CG'),
            'len': WINDOW
        }
        # Handle the first window as a special case (no previous window to diff against)
        if isCpG(content):
            fulfill.append(True)
        else:
            fulfill.append(False)

        # Slide the window one base at a time, updating content incrementally
        for i in range(1, len(seq) - WINDOW + 1):
          poped = seq[i-1:i+1] # base leaving the window + its neighbor
          pushed = seq[i + WINDOW - 2: i + WINDOW] # base entering the window + its neighbor
          alteration = content_alter(poped, pushed) # returns changes of values
          content['c'] += alteration['c']
          content['g'] += alteration['g']
          content['cpg'] += alteration['cpg']

          # check if this window is a CpG island
          if isCpG(content):
              fulfill.append(True)
          else:
              fulfill.append(False)

        # results: list of [start, end] window-index pairs passing the
        # length/content criteria (island_finder does not itself re-verify
        # obs/exp on the merged region — that final check happens below)
        results = island_finder(fulfill)

        if results is None:
            continue
        for result in results:
            length = result[1] + WINDOW - result[0]
            target = seq[result[0]:result[1] + WINDOW]
            target_content = {
                'c': target.count('C'),
                'g': target.count('G'),
                'cpg': target.count('CG'),
                'len': length
            }

            if isCpG(target_content):
                temp_islands.append([
                    entry.id,
                    entry.chrom,
                    result[0],
                    result[1]+WINDOW, #For loops that does not include the last number
                    target,
                    (target_content['c'] + target_content['g']) / length,
                    length,
                    (target_content['cpg'] * length) / (target_content['c'] * target_content['g'])
                ])
    return temp_islands

In [ ]:
def save_csv():
  """
  Load upstream sequence CSVs (from downloader.ipynb) and run CpG island
  detection for all three species, saving results to dataset/processed/.
  """
  arab_csv = pd.read_csv('../dataset/processed/arabidopsis_mrna.csv')
  must_csv = pd.read_csv('../dataset/processed/mustard_mrna.csv')
  cabb_csv = pd.read_csv('../dataset/processed/c_cabbage_mrna.csv')

  csv_columns = ['id', 'chrom', 'start', 'end', 'seq', 'gc-content', 'length', 'obs/exp']

  print('Start finding CpG islands.')
  arab_islands = pd.DataFrame(findCpG(arab_csv), columns=csv_columns)
  print('Ended finding arabidopsis CpG islands.')
  must_islands = pd.DataFrame(findCpG(must_csv), columns=csv_columns)
  print('Ended finding mustard CpG islands.')
  cabb_islands = pd.DataFrame(findCpG(cabb_csv), columns=csv_columns)
  print('Ended finding Chinese cabbage CpG islands.')

  arab_islands.to_csv("../dataset/processed/arabidopsis_islands.csv", index=False)
  must_islands.to_csv("../dataset/processed/mustard_islands.csv", index=False)
  cabb_islands.to_csv("../dataset/processed/c_cabbage_islands.csv", index=False)

  print("Saved CpG island datasets at '../dataset/processed'.")

In [ ]:
save_csv()

Start finding CpG islands.


0it [00:00, ?it/s]

Ended finding arabidopsis CpG islands.


0it [00:00, ?it/s]

Ended finding mustard CpG islands.


0it [00:00, ?it/s]

Ended finding Chinese cabbage CpG islands.
Saved CpG island datasets at '../dataset/processed'.
